# Phase 4 - Baseline ML Models for RUL Prediction (CMAPSS FD001)

**Goal:** Train, tune, and compare four models on the flattened sliding-window features
produced by the Phase 3 Preprocessor:
Linear Regression (baseline), Random Forest, XGBoost, and LightGBM.

**Workflow:**
1. Load pre-processed flat features from `data/processed/`
2. Evaluate with `GroupKFold` (group = engine ID) to prevent leakage across folds
3. Tune RF / XGBoost / LightGBM with Optuna (50 trials, TPE sampler, objective = CV RMSE)
4. Report RMSE / MAE / R2 on the held-out validation split and the true test RUL (uncapped)
5. Error analysis focused on the safety-critical low-RUL region (< 30 cycles)
6. Save comparison table to `reports/` and identify the best model for Stage 2

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
from lightgbm import LGBMRegressor
from matplotlib.patches import Patch
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold
from xgboost import XGBRegressor

from rul_prediction.config.configuration import load_config
from rul_prediction.components.data_ingestion import DataIngestion

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
%matplotlib inline

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('seaborn-whitegrid')

In [ ]:
import os

# Resolve project root (one level up from notebooks/) and set as working directory
# os.chdir ensures relative paths in DataIngestion work regardless of kernel start dir
PROJECT_ROOT = Path('..').resolve()
os.chdir(PROJECT_ROOT)

config = load_config()
SEED = config['seed']

PROCESSED_DIR = PROJECT_ROOT / config['paths']['processed_data_dir']
REPORTS_DIR = PROJECT_ROOT / 'reports'
FIGURES_DIR = REPORTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Processed dir: {PROCESSED_DIR}')
print(f'Seed         : {SEED}')

## Section 1 - Load Preprocessed Data

Flat features (`X_flat`) are parquet files with columns `{sensor}_t{step}` (510 features = 17 sensors x 30 time steps).  
Targets (`y`) are RUL **capped at 125** for train/val; test targets are uncapped ground truth.

In [ ]:
X_train = pd.read_parquet(PROCESSED_DIR / 'train_X_flat.parquet')
y_train = np.load(PROCESSED_DIR / 'train_y.npy')
engine_ids_train = np.load(PROCESSED_DIR / 'train_engine_ids.npy')

X_val = pd.read_parquet(PROCESSED_DIR / 'val_X_flat.parquet')
y_val = np.load(PROCESSED_DIR / 'val_y.npy')
engine_ids_val = np.load(PROCESSED_DIR / 'val_engine_ids.npy')

X_test = pd.read_parquet(PROCESSED_DIR / 'test_X_flat.parquet')
engine_ids_test = np.load(PROCESSED_DIR / 'test_engine_ids.npy')

print(f'Train: X={X_train.shape}, y={y_train.shape}, engines={np.unique(engine_ids_train).size}')
print(f'Val  : X={X_val.shape}, y={y_val.shape}, engines={np.unique(engine_ids_val).size}')
print(f'Test : X={X_test.shape}, engines={np.unique(engine_ids_test).size}')

In [ ]:
# Convert to numpy for sklearn indexing (GroupKFold requires array indexing)
X_train_np = X_train.to_numpy(dtype=np.float32)
X_val_np   = X_val.to_numpy(dtype=np.float32)

print(f'X_train_np: {X_train_np.shape}, dtype={X_train_np.dtype}')
print(f'y_train range: [{y_train.min():.0f}, {y_train.max():.0f}], mean={y_train.mean():.1f}')

## Section 2 - Load Test Ground-Truth RUL

`RUL_FD001.txt` has **one uncapped RUL value per test engine** (100 engines).  
The Preprocessor creates sliding windows over each test engine sequence, so `X_test` has
multiple rows per engine. We evaluate only on the **last window of each engine** to align
with the ground-truth labels.

In [ ]:
subset = config['dataset']['active_subset']
test_rul_df = DataIngestion(subset=subset).load_test_rul()
y_test = test_rul_df['RUL'].values  # uncapped, shape (100,)

# Last window per test engine aligns with RUL_FD001.txt ordering
unique_test_engines = np.unique(engine_ids_test)
last_idx = [int(np.where(engine_ids_test == eid)[0][-1]) for eid in unique_test_engines]
X_test_eval    = X_test.iloc[last_idx].reset_index(drop=True)
X_test_eval_np = X_test_eval.to_numpy(dtype=np.float32)

assert len(X_test_eval_np) == len(y_test), (
    f'Alignment mismatch: {len(X_test_eval_np)} windows vs {len(y_test)} ground-truth RULs'
)
print(f'Test eval: X={X_test_eval_np.shape}, y_test={y_test.shape}')
print(f'Test RUL  range: [{y_test.min()}, {y_test.max()}], median={np.median(y_test):.1f}')

## Section 3 - GroupKFold Cross-Validation Strategy

**Why not plain K-Fold?**  
The Preprocessor generates many overlapping sliding windows per engine. If plain K-Fold
is used, windows from the *same engine* appear in both the training fold and the validation fold.
Since adjacent windows share 29 of 30 time steps, the model effectively sees the
validation data during training, inflating CV scores.

**GroupKFold** (group = `engine_id`) ensures every engine's windows land entirely in
either the training fold or the validation fold, never both. This gives a realistic
estimate of how well the model generalises to engines it has never seen.

In [ ]:
def cv_evaluate(model_factory, X, y, groups, n_splits=5):
    """GroupKFold CV returning mean/std RMSE, MAE, R2 across folds."""
    gkf = GroupKFold(n_splits=n_splits)
    rmse_list, mae_list, r2_list = [], [], []
    for train_idx, val_idx in gkf.split(X, y, groups):
        model = model_factory()
        model.fit(X[train_idx], y[train_idx])
        y_pred = model.predict(X[val_idx])
        rmse_list.append(float(np.sqrt(mean_squared_error(y[val_idx], y_pred))))
        mae_list.append(float(mean_absolute_error(y[val_idx], y_pred)))
        r2_list.append(float(r2_score(y[val_idx], y_pred)))
    return {
        'rmse_mean': float(np.mean(rmse_list)),
        'rmse_std' : float(np.std(rmse_list)),
        'mae_mean' : float(np.mean(mae_list)),
        'r2_mean'  : float(np.mean(r2_list)),
    }


def eval_split(model, X, y_true):
    """RMSE, MAE, R2 on a single data split."""
    y_pred = model.predict(X)
    return {
        'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'mae' : float(mean_absolute_error(y_true, y_pred)),
        'r2'  : float(r2_score(y_true, y_pred)),
    }


results = []          # comparison table rows
trained_models = {}   # {model_name: fitted_model}

## Section 4 - Linear Regression (Baseline)

No hyperparameter tuning needed. Provides a linear lower bound on performance.

In [ ]:
lr = LinearRegression()
lr.fit(X_train_np, y_train)

lr_cv  = cv_evaluate(LinearRegression, X_train_np, y_train, engine_ids_train)
lr_val  = eval_split(lr, X_val_np, y_val)
lr_test = eval_split(lr, X_test_eval_np, y_test)

print('Linear Regression')
print(f'  CV   RMSE: {lr_cv["rmse_mean"]:.2f} +/- {lr_cv["rmse_std"]:.2f}')
print(f'  Val  RMSE={lr_val["rmse"]:.2f}  MAE={lr_val["mae"]:.2f}  R2={lr_val["r2"]:.3f}')
print(f'  Test RMSE={lr_test["rmse"]:.2f}  MAE={lr_test["mae"]:.2f}  R2={lr_test["r2"]:.3f}')

results.append({
    'model'       : 'LinearRegression',
    'cv_rmse_mean': lr_cv['rmse_mean'],
    'cv_rmse_std' : lr_cv['rmse_std'],
    'val_rmse'    : lr_val['rmse'],
    'val_mae'     : lr_val['mae'],
    'val_r2'      : lr_val['r2'],
    'test_rmse'   : lr_test['rmse'],
    'test_mae'    : lr_test['mae'],
    'test_r2'     : lr_test['r2'],
})
trained_models['LinearRegression'] = lr

## Section 5 - Random Forest + Optuna HPO

Optuna uses the TPE (Tree-structured Parzen Estimator) sampler to adaptively explore
the hyperparameter space. Each trial evaluates a candidate config via 5-fold GroupKFold CV
and minimises mean CV RMSE.

In [ ]:
def objective_rf(trial):
    params = {
        'n_estimators'    : trial.suggest_int('n_estimators', 100, 500),
        'max_depth'       : trial.suggest_int('max_depth', 5, 25),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features'    : trial.suggest_categorical('max_features', ['sqrt', 'log2']),
    }
    result = cv_evaluate(
        lambda: RandomForestRegressor(**params, n_jobs=-1, random_state=SEED),
        X_train_np, y_train, engine_ids_train
    )
    return result['rmse_mean']


rf_study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
rf_study.optimize(objective_rf, n_trials=50)

print(f'RF best CV RMSE : {rf_study.best_value:.2f}')
print(f'RF best params  : {rf_study.best_params}')

In [ ]:
best_rf_params = rf_study.best_params

# Re-run CV with best params to get fold-level std
rf_cv = cv_evaluate(
    lambda: RandomForestRegressor(**best_rf_params, n_jobs=-1, random_state=SEED),
    X_train_np, y_train, engine_ids_train
)

best_rf = RandomForestRegressor(**best_rf_params, n_jobs=-1, random_state=SEED)
best_rf.fit(X_train_np, y_train)

rf_val  = eval_split(best_rf, X_val_np, y_val)
rf_test = eval_split(best_rf, X_test_eval_np, y_test)

print('Random Forest (Optuna best)')
print(f'  CV   RMSE: {rf_cv["rmse_mean"]:.2f} +/- {rf_cv["rmse_std"]:.2f}')
print(f'  Val  RMSE={rf_val["rmse"]:.2f}  MAE={rf_val["mae"]:.2f}  R2={rf_val["r2"]:.3f}')
print(f'  Test RMSE={rf_test["rmse"]:.2f}  MAE={rf_test["mae"]:.2f}  R2={rf_test["r2"]:.3f}')

results.append({
    'model'       : 'RandomForest',
    'cv_rmse_mean': rf_cv['rmse_mean'],
    'cv_rmse_std' : rf_cv['rmse_std'],
    'val_rmse'    : rf_val['rmse'],
    'val_mae'     : rf_val['mae'],
    'val_r2'      : rf_val['r2'],
    'test_rmse'   : rf_test['rmse'],
    'test_mae'    : rf_test['mae'],
    'test_r2'     : rf_test['r2'],
})
trained_models['RandomForest'] = best_rf

## Section 6 - XGBoost + Optuna HPO

In [ ]:
def objective_xgb(trial):
    params = {
        'n_estimators'    : trial.suggest_int('n_estimators', 100, 600),
        'max_depth'       : trial.suggest_int('max_depth', 3, 10),
        'learning_rate'   : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample'       : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
    }
    result = cv_evaluate(
        lambda: XGBRegressor(**params, device='cpu', tree_method='hist'),
        X_train_np, y_train, engine_ids_train
    )
    return result['rmse_mean']


xgb_study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
xgb_study.optimize(objective_xgb, n_trials=50)

print(f'XGBoost best CV RMSE : {xgb_study.best_value:.2f}')
print(f'XGBoost best params  : {xgb_study.best_params}')

In [ ]:
best_xgb_params = xgb_study.best_params

xgb_cv = cv_evaluate(
    lambda: XGBRegressor(**best_xgb_params, device='cpu', tree_method='hist'),
    X_train_np, y_train, engine_ids_train
)

best_xgb = XGBRegressor(**best_xgb_params, device='cpu', tree_method='hist')
best_xgb.fit(X_train_np, y_train)

xgb_val  = eval_split(best_xgb, X_val_np, y_val)
xgb_test = eval_split(best_xgb, X_test_eval_np, y_test)

print('XGBoost (Optuna best)')
print(f'  CV   RMSE: {xgb_cv["rmse_mean"]:.2f} +/- {xgb_cv["rmse_std"]:.2f}')
print(f'  Val  RMSE={xgb_val["rmse"]:.2f}  MAE={xgb_val["mae"]:.2f}  R2={xgb_val["r2"]:.3f}')
print(f'  Test RMSE={xgb_test["rmse"]:.2f}  MAE={xgb_test["mae"]:.2f}  R2={xgb_test["r2"]:.3f}')

results.append({
    'model'       : 'XGBoost',
    'cv_rmse_mean': xgb_cv['rmse_mean'],
    'cv_rmse_std' : xgb_cv['rmse_std'],
    'val_rmse'    : xgb_val['rmse'],
    'val_mae'     : xgb_val['mae'],
    'val_r2'      : xgb_val['r2'],
    'test_rmse'   : xgb_test['rmse'],
    'test_mae'    : xgb_test['mae'],
    'test_r2'     : xgb_test['r2'],
})
trained_models['XGBoost'] = best_xgb

## Section 7 - LightGBM + Optuna HPO

In [ ]:
def objective_lgbm(trial):
    params = {
        'n_estimators'    : trial.suggest_int('n_estimators', 100, 600),
        'max_depth'       : trial.suggest_int('max_depth', 3, 10),
        'learning_rate'   : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves'      : trial.suggest_int('num_leaves', 20, 150),
        'subsample'       : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
    }
    result = cv_evaluate(
        lambda: LGBMRegressor(**params, n_jobs=-1, verbosity=-1),
        X_train_np, y_train, engine_ids_train
    )
    return result['rmse_mean']


lgbm_study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
lgbm_study.optimize(objective_lgbm, n_trials=50)

print(f'LightGBM best CV RMSE : {lgbm_study.best_value:.2f}')
print(f'LightGBM best params  : {lgbm_study.best_params}')

In [ ]:
best_lgbm_params = lgbm_study.best_params

lgbm_cv = cv_evaluate(
    lambda: LGBMRegressor(**best_lgbm_params, n_jobs=-1, verbosity=-1),
    X_train_np, y_train, engine_ids_train
)

best_lgbm = LGBMRegressor(**best_lgbm_params, n_jobs=-1, verbosity=-1)
best_lgbm.fit(X_train_np, y_train)

lgbm_val  = eval_split(best_lgbm, X_val_np, y_val)
lgbm_test = eval_split(best_lgbm, X_test_eval_np, y_test)

print('LightGBM (Optuna best)')
print(f'  CV   RMSE: {lgbm_cv["rmse_mean"]:.2f} +/- {lgbm_cv["rmse_std"]:.2f}')
print(f'  Val  RMSE={lgbm_val["rmse"]:.2f}  MAE={lgbm_val["mae"]:.2f}  R2={lgbm_val["r2"]:.3f}')
print(f'  Test RMSE={lgbm_test["rmse"]:.2f}  MAE={lgbm_test["mae"]:.2f}  R2={lgbm_test["r2"]:.3f}')

results.append({
    'model'       : 'LightGBM',
    'cv_rmse_mean': lgbm_cv['rmse_mean'],
    'cv_rmse_std' : lgbm_cv['rmse_std'],
    'val_rmse'    : lgbm_val['rmse'],
    'val_mae'     : lgbm_val['mae'],
    'val_r2'      : lgbm_val['r2'],
    'test_rmse'   : lgbm_test['rmse'],
    'test_mae'    : lgbm_test['mae'],
    'test_r2'     : lgbm_test['r2'],
})
trained_models['LightGBM'] = best_lgbm

## Section 8 - Model Comparison

Ranked by test RMSE (primary metric). The model with lowest test RMSE is carried forward
to Stage 2 productionisation. Copy its best hyperparameters to `configs/config.yaml`.

In [ ]:
comparison_df = (
    pd.DataFrame(results)
    .sort_values('test_rmse')
    .reset_index(drop=True)
)

# Format for display
display_df = comparison_df.copy()
for col in ['cv_rmse_mean', 'cv_rmse_std', 'val_rmse', 'val_mae', 'test_rmse', 'test_mae']:
    display_df[col] = display_df[col].map('{:.2f}'.format)
for col in ['val_r2', 'test_r2']:
    display_df[col] = display_df[col].map('{:.3f}'.format)

print(display_df.to_string(index=False))

comparison_df.to_csv(REPORTS_DIR / 'model_comparison.csv', index=False)
print(f'\nSaved to {REPORTS_DIR / "model_comparison.csv"}')

best_model_name = comparison_df.iloc[0]['model']
best_model      = trained_models[best_model_name]
print(f'\nBest model: {best_model_name}  (test RMSE={comparison_df.iloc[0]["test_rmse"]:.2f})')

## Section 9 - Error Analysis (Safety-Critical Focus)

The low-RUL zone (< 30 cycles remaining) is safety-critical for aviation: if the model
**over-predicts** RUL here (says the engine is healthier than it is), maintenance may be
delayed, risking in-flight failure. **Under-prediction** is conservative and safe but
causes unnecessary early removals.

We examine two plots for the best model on the test set:
1. **Predicted vs True RUL** - coloured by zone (low / mid / high)
2. **Residuals vs True RUL** - a positive slope at low RUL reveals systematic over-prediction near failure

In [ ]:
y_pred_test  = best_model.predict(X_test_eval_np)
residuals    = y_pred_test - y_test
zone_colors  = ['red' if r < 30 else ('orange' if r < 80 else 'steelblue') for r in y_test]

legend_elements = [
    Patch(facecolor='red',       label='Low RUL < 30 (safety-critical)'),
    Patch(facecolor='orange',    label='Mid RUL 30-80'),
    Patch(facecolor='steelblue', label='High RUL > 80'),
    plt.Line2D([0], [0], color='black', linestyle='--', label='Perfect prediction'),
]

# --- Plot 1: Predicted vs True RUL ---
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test, y_pred_test, c=zone_colors, alpha=0.7, s=40, edgecolors='none')
max_rul = max(float(y_test.max()), float(y_pred_test.max())) + 10
ax.plot([0, max_rul], [0, max_rul], 'k--', lw=1.5)
ax.set_xlabel('True RUL (cycles)')
ax.set_ylabel('Predicted RUL (cycles)')
ax.set_title(f'{best_model_name} - Predicted vs True RUL (test set)')
ax.legend(handles=legend_elements, fontsize=9)
plt.tight_layout()
scatter_path = FIGURES_DIR / f'model_scatter_{best_model_name.lower()}.png'
plt.savefig(scatter_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {scatter_path}')

# --- Plot 2: Residuals vs True RUL ---
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(y_test, residuals, c=zone_colors, alpha=0.7, s=40, edgecolors='none')
ax.axhline(0, color='black', linestyle='--', lw=1.5)
ax.set_xlabel('True RUL (cycles)')
ax.set_ylabel('Residual = Predicted - True (cycles)')
ax.set_title(f'{best_model_name} - Residuals vs True RUL (test set)')
ax.legend(handles=legend_elements[:3], fontsize=9)
plt.tight_layout()
residual_path = FIGURES_DIR / f'model_residuals_{best_model_name.lower()}.png'
plt.savefig(residual_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {residual_path}')

# Low-RUL region summary
low_mask = y_test < 30
if low_mask.sum() > 0:
    low_rmse = float(np.sqrt(mean_squared_error(y_test[low_mask], y_pred_test[low_mask])))
    low_bias = float(residuals[low_mask].mean())
    print(f'\nLow-RUL region (< 30 cycles, n={low_mask.sum()})')
    print(f'  RMSE  : {low_rmse:.2f}')
    print(f'  Bias  : {low_bias:+.2f}  (positive = over-predicts RUL, dangerous)')

## Section 10 - Key Findings

### Model Comparison Summary
*(See `reports/model_comparison.csv` and the comparison table above for exact numbers.)*

### Error Analysis Observations
- **Low-RUL zone (< 30 cycles):** Check the bias value printed above. A positive bias
  means the model over-predicts RUL near failure — the most dangerous failure mode.
- **Scatter plot:** Points above the diagonal line are over-predictions; points below are
  under-predictions. The red zone should ideally cluster below or on the diagonal.
- **Residual plot:** A downward slope from right to left (as true RUL decreases) indicates
  the model under-estimates degradation severity near failure.

### Stage 2 Action Items
1. Copy the best model name and its Optuna best params to `configs/config.yaml` under
   the `model:` section (already added as a placeholder).
2. Run `uv run python -m rul_prediction.components.model_trainer` to train and persist
   the production model via `ModelTrainer`.
3. The saved `.pkl` in `saved_models/` is the artifact carried forward to Phase 5
   (deep learning comparison) and Phase 6 (SHAP explainability).